# Data access

In [ ]:
"""
Get A specific day of global data from the dataset, from the S3 bucket.
"""
from datetime import date
from wekeo_combined_chain.s3_access import get_combined_ds

day = date(2025, 8, 2)
ds = get_combined_ds(day)

In [ ]:
"""
Select a specific area of interest in the dataset.
"""

areas = {
    "Global":          [ 90., -90.,  180., -180.],
    "North_America":   [ 90.,   9.,  -20., -169.],
    "South_America":   [  9., -60.,  -20.,  120.],
    "Europe":          [ 90.,  36.,   31.,  -20.],
    "Africa":          [ 36., -60.,   60.,  -20.],
    "Russia":          [ 90.,  36., -169.,   31.],
    "Asia":            [ 36., -10., -169.,   60.],
    "Australia":       [-10., -60., -120.,   60.],
    # "Central_Pacific": [  9., -10., -120., -169.],
    # "Antarctic":       [-60., -90.,  180., -180.],
}
    
from wekeo_combined_chain.utils import select_area

area_name = "Global"
area = areas[area_name]

## User override example:
# area_name = "User defined: France"
# area = [53., 41., 9., -5.]
    
ds_area = select_area(ds, area)

In [ ]:
from wekeo_combined_chain import postprocess

ds_post, df_plumes = postprocess.compute(ds_area)

In [ ]:
from wekeo_combined_chain.postprocess import table as T

#print("Plumes:")
print("-" * 75)
#print(T.plume_table(df_plumes))
# MAJ 11/06/26
T.display_plume_tables(df_plumes)

print()
#print("Tiny plumes:")
print("-" * 75)
# MAJ 11/06/26
#print(T.tiny_plume_table(df_plumes))
T.display_tiny_plume_tables(df_plumes)

In [ ]:
from wekeo_combined_chain.postprocess import plot as P
from pathlib import Path

date_str = day.strftime("%Y%m%d")
output_dir = Path("output") / day.strftime("%Y_%m_%d") / area_name
output_dir.mkdir(parents=True, exist_ok=True)


## Map 1 — Plumes × FRP overlay

In [ ]:
# Toggle save_to to write to disk, or leave None for inline display only
P.plot_plumes_frp(ds_area, ds_post, date_str, frp_channel="SWIR",
                  save_to=output_dir / f"plumes_frp_SWIR_{date_str}.png")


## Map 2 — Fire score per plume

In [ ]:
P.plot_fire_score_plume(ds_area, ds_post, df_plumes, date_str, band="MWIR",
                        save_to=output_dir / f"fire_score_plume_MWIR_{date_str}.png")


## Map 3 — Fire score per pixel

In [ ]:
P.plot_fire_score_pixel(ds_area, ds_post, date_str, band="MWIR",
                        save_to=output_dir / f"fire_score_pixel_SWIR_{date_str}.png")
    

## Map 4 — Plume envelopes + source confidence

In [ ]:
P.plot_plume_envelopes(ds_area, ds_post, df_plumes, date_str, frp_channel="MWIR",
                       save_to=output_dir / f"plume_envelopes_MWIR_{date_str}.png")


## Animations over the last n days (default: n = 5)

In [ ]:
"""
Get A specific day of global data from the dataset, from the S3 bucket.
"""
from datetime import date, timedelta
from wekeo_combined_chain.s3_access import get_combined_ds_range

# dataset over the n+1 days
n = 5 # number of days analysed
band = 'MWIR'
lst_dates = [day - timedelta(days=i) for i in range(1, n+1)][::-1] # dates list
start = lst_dates[0]
end = day

dsp = get_combined_ds_range(start, end)


In [ ]:
"""
Select a specific area of interest in the dataset.
"""

area = areas[area_name]

dsp = select_area(dsp, area)

In [ ]:
"""
Coarsen dataset to reduce spatial resolution by block-averaging.
"""

FACTOR = 8 # 12-16 good on a global scale, for regionnal smaller factor advised, around 4-8
dspc = dsp.coarsen(latitude=FACTOR, longitude=FACTOR, boundary="trim").mean()

## Animation 1: S5P-PCA mean score CO

In [ ]:
from wekeo_combined_chain.timeseries.plot_maps import animate_score_CO_map
from datetime import  timedelta

animate_score_CO_map(dspc)

## Animation 2: S5P-PCA Plume presence

In [ ]:
from wekeo_combined_chain.timeseries.plot_maps import animate_plume_map

# Interactive slider — scrub through days
animate_plume_map(dspc)